
# **Pan-sharpening**

เทคนิค Pan-sharpening เป็นเทคนิคที่ใช้กับภาพถ่ายดาวเทียมที่มี Panchromatic band เช่น Landsat-8 และ Landsat-9 
เพื่อเพิ่มความละเอียดเชิงพื้นที่ของภาพสี (Multispectral) ให้คมชัดมากขึ้น

Lab นี้จะแสดงตัวอย่างการทำ Pan-sharpening ด้วยข้อมูล Landsat-9 บน Google Earth Engine


In [ ]:

import ee
import geemap

ee.Authenticate()
ee.Initialize()


In [ ]:

# กำหนดพื้นที่สนใจ (นครปฐม)
geometry = ee.Geometry.Point([100.0615, 13.8199])

# เรียกภาพ Landsat 9
image = (ee.ImageCollection("LANDSAT/LC09/C02/T1_TOA")
         .filterDate('2024-01-01', '2024-12-31')
         .filterBounds(geometry)
         .sort('CLOUD_COVER')
         .first())

# RGB และ Panchromatic
rgb = image.select(['B4', 'B3', 'B2'])
pan = image.select('B8')


In [ ]:

# ทำ Pan-sharpening ด้วย HSV method

huesat = rgb.rgbToHsv().select(['hue', 'saturation'])
upres = ee.Image.cat([huesat, pan]).hsvToRgb()

panSharpened = upres


In [ ]:

Map = geemap.Map(center=[13.8199, 100.0615], zoom=10)

vis = {'min': 0, 'max': 0.3}

Map.addLayer(rgb, vis, 'Original RGB')
Map.addLayer(panSharpened, vis, 'Pan-sharpened RGB')

Map



## คำถาม

**1. เป้าหมายหลักของ Pan-sharpening คืออะไร**  
เพื่อเพิ่มความละเอียดเชิงพื้นที่ของภาพ Multispectral โดยใช้ข้อมูลจาก Panchromatic band ทำให้ภาพมีรายละเอียดคมชัดมากขึ้น

**2. ข้อดีของการใช้ Landsat-9 ในการทำ Pan-sharpening**  
มี Panchromatic band ความละเอียด 15 เมตร ทำให้สามารถเพิ่มรายละเอียดของภาพ RGB ได้

**3. ความแตกต่างระหว่างภาพก่อนและหลัง Pan-sharpening**  
ภาพหลัง Pan-sharpening จะมีความคมชัดมากขึ้น เห็นรายละเอียดพื้นที่ชัดเจนกว่าเดิม

**4. ข้อจำกัดของ Pan-sharpening**  
อาจทำให้ค่าทางสเปกตรัมผิดเพี้ยน และไม่เหมาะกับงานวิเคราะห์เชิงปริมาณบางประเภท
